# MM4250 Driver Example

A walkthrough of `drivers/MM4250_finalized.py`, the QCoDeS driver for the **Menlo Micro MM4250** SP6T cryogenic RF switch, driven over USB HID through its HiV Driver Board. It covers connecting, selecting RF channels, the internal calibration states, reconnecting without disturbing the switch, and recording the switch position alongside your data.

**This drives real hardware.** Every cell that sets `channel` or `state` physically moves the switch. Don't run it while a sweep in another kernel is using the switch.

**Before running:**
- `pip install hidapi` (not `hid`, which is a different package with a same-named module).
- The HiV Driver Board is plugged in over USB and powered.
- No other kernel has the switch open. The board takes an exclusive USB handle, so close any other notebook that created an `MM4250` first.

## Imports

Walks up from this notebook's folder until it finds `drivers/` (the repo root) and imports the driver from there.

In [ ]:
import sys
from pathlib import Path

try:
    here = Path(_dh[0])  # Jupyter sets _dh[0] to this notebook's directory
except NameError:
    here = Path.cwd()

root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print(f"drivers from: {root}")

from qcodes.station import Station

from drivers.MM4250_finalized import MM4250, SP6TState

## Connect

Opens the driver board over USB and prints the standard QCoDeS connect message.

Connecting **forces `ALL_OPEN`** by default, so the switch always starts from a known, de-energized state. To reconnect without moving the switch, see *Reconnecting mid-experiment* below.

In [ ]:
switch = MM4250("switch")

## What the driver exposes

Two parameters, both views of the same switch position:

- `state`: the full position: `"ALL_OPEN"`, `"RFC_RF1"`...`"RFC_RF6"`, `"INTERNAL_LOAD"` or `"INTERNAL_SHORT"`.
- `channel`: the friendlier 1-6 view, matching the RF1-RF6 SMA ports.

Getting either one reads the position back from the board rather than returning what was last set.

The snapshot reads `channel` too, so while the switch is on `ALL_OPEN` you'll see a "not a numbered RF channel" warning here. That's expected.

In [ ]:
switch.print_readable_snapshot(update=True)

## Select an RF channel

`switch.channel(n)` connects RFC to RFn. Reading `channel` or `state` afterwards confirms the board actually took the command.

In [ ]:
switch.channel(3)
print("channel:", switch.channel())
print("state:  ", switch.state())

## Step through all six channels

The pattern a sweep uses: set each channel and verify the readback before measuring. The driver already waits the datasheet's typical switching time (25 ms) after each command.

In [ ]:
for ch in range(1, 7):
    switch.channel(ch)
    readback = switch.channel()
    assert readback == ch, f"asked for RF{ch}, board reports {readback}"
    print(f"RF{ch}: ok ({switch.state()})")

## Internal calibration states

`ALL_OPEN`, `INTERNAL_LOAD` and `INTERNAL_SHORT` aren't numbered ports, so they're only reachable through `state`. On one of them, `channel` returns `None` and logs a warning instead of guessing.

In [ ]:
switch.state("INTERNAL_LOAD")
print("state:  ", switch.state())
print("channel:", switch.channel())

You can also pass the `SP6TState` enum instead of a string, which lets your editor autocomplete the names:

In [ ]:
switch.state(SP6TState.INTERNAL_SHORT)
switch.state()

## Which HV lines are energized

`active_hv_lines()` lists the driver-board HV lines that should be energized for the current position, per the datasheet. Useful when checking the board's outputs with a meter or scope.

In [ ]:
switch.channel(1)
switch.active_hv_lines()

## Bad inputs never reach the hardware

Out-of-range channels and misspelled states are rejected by the parameter validators before anything is sent to the board.

In [ ]:
for bad in [lambda: switch.channel(7), lambda: switch.state("RFC_RF7")]:
    try:
        bad()
    except ValueError as err:
        print("rejected:", err)

print("still on:", switch.state())

## Reconnecting mid-experiment

If you close the switch or restart the kernel partway through, reconnect with `reset_on_connect=False` to leave the switch where it is. The driver reads the current position from the board instead of forcing `ALL_OPEN`.

In [ ]:
switch.channel(5)
switch.close()

switch = MM4250("switch", reset_on_connect=False)
print("channel after reconnect:", switch.channel())

Creating a second `MM4250` with a name that's already in use raises an error **without** touching the switch, so accidentally re-running the connect cell can't reset a sweep in progress:

In [ ]:
try:
    MM4250("switch")
except KeyError as err:
    print("refused:", err)

print("still on:", switch.state())

## Recording the switch position with your data

Add the switch to a QCoDeS `Station`. Every dataset measured with that station saves a snapshot of it, so each run records which channel it was taken on without you logging it by hand.

In [ ]:
station = Station(switch)
station.snapshot(update=True)["instruments"]["switch"]["parameters"]["state"]["value"]

## Clean up

Return the switch to `ALL_OPEN` and release the USB handle so other kernels can connect.

In [ ]:
switch.open_all()
switch.close()